# V-JEPA 2 + Decoder — Ball Occlusion Inference

**Run on Kaggle T4 GPU.**  Requires:
- Kaggle dataset `ball-clips` uploaded from `data/processed/ball_clips/` on your Mac
- Kaggle Secret `HF_TOKEN` set to your HuggingFace token (V-JEPA 2 weights + decoder download automatically)

Pipeline per video clip:
```
Causal sliding window: for each of N_EVAL evaluation points t,
  use only frames 0..t (no future leakage)
  Spatial mask: patch columns 8–15 excluded (pixels 128–256)
  Window of 16 frames → V-JEPA 2.1 Encoder + Predictor
  Last temporal position (frames t-1, t) → [1408, 24, 24]
  → LocalizationDecoder → heatmap [1, 384, 384]
```

## 0. Install V-JEPA 2 from Source

In [ ]:
# import subprocess, sys
# r = subprocess.run('git clone --depth 1 https://github.com/facebookresearch/vjepa2 /tmp/vjepa2 2>&1 || echo already_cloned',
#                    shell=True, capture_output=True, text=True)
# print(r.stdout)
# r = subprocess.run(f'{sys.executable} -m pip install -e /tmp/vjepa2 -q',
#                    shell=True, capture_output=True, text=True)
# print(r.stdout[-200:] if r.stdout else 'ok')

## 1. Imports & Constants

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from huggingface_hub import hf_hub_download, login

# HuggingFace auth (Kaggle → Add-ons → Secrets → HF_TOKEN)
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)
    print('HuggingFace login OK')
except Exception as e:
    print(f'HF login skipped ({e})')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    cap = torch.cuda.get_device_capability()
    if cap[0] < 7:
        print(f'WARNING: GPU capability {cap} < 7.0 — switch to T4'); DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}')

# V-JEPA 2 ViT-G/16 @ 384

IMAGE_SIZE = 384
PATCH_GRID = 24
N_SPATIAL  = 576    # 24×24
N_TOTAL    = 18432  # 32×576
EMBED_DIM  = 1408
PATCH_SIZE = 16

NUM_FRAMES   = 64
TUBELET_SIZE = 2
N_TEMPORAL   = NUM_FRAMES // TUBELET_SIZE  # 32

N_EVAL = 8  # causal evaluation points per clip

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

# import kagglehub
# path = kagglehub.dataset_download("binhminhvu/masked-ball-flying-dataset")
# print(path)   # prints where it was saved, e.g. /root/.cache/kagglehub/datasets/...


BALL_CLIPS_DIR = Path('/root/.cache/kagglehub/datasets/binhminhvu/masked-ball-flying-dataset/versions/1/ball_clips')
OUT_DIR        = Path('../inference_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Download V-JEPA 2 weights
print('Downloading V-JEPA 2 weights ...')
VJEPA_WEIGHTS = Path('../vjepa2_1_vitg_384.pt')
print(f'V-JEPA 2 weights: {VJEPA_WEIGHTS}')

# Download trained decoder
print('Downloading decoder checkpoint ...')
# DECODER_CKPT = Path(hf_hub_download(
#     repo_id='Bmingg/cs280-localization-decoder-vjepa21-vitg-384',
#     filename='decoder_best.pt',
#     local_dir='/kaggle/working',
# ))
DECODER_CKPT = Path("../outputs/checkpoints/decoder_epoch020.pt")

print(f'Decoder checkpoint: {DECODER_CKPT}')

print(f'Token layout: {N_TEMPORAL} temporal × {N_SPATIAL} spatial = {N_TOTAL} total')

## 2. Mask Definition

The spatial mask covers patch **columns 5–10** (inclusive) across **all temporal positions**,
corresponding to pixels x∈[80, 176] — the center-right third of the 256×256 frame.

Token index for (temporal=t, row=r, col=c): `t * N_SPATIAL + r * PATCH_GRID + c`

In [ ]:
MASK_COL_START = 8
MASK_COL_END   = 15   # inclusive  (pixels 80–176 out of 256)

context_indices = []
target_indices  = []

for t in range(N_TEMPORAL):
    for r in range(PATCH_GRID):
        for c in range(PATCH_GRID):
            idx = t * N_SPATIAL + r * PATCH_GRID + c
            if MASK_COL_START <= c <= MASK_COL_END:
                target_indices.append(idx)
            else:
                context_indices.append(idx)

ctx_t = torch.tensor(context_indices, dtype=torch.long)
tgt_t = torch.tensor(target_indices,  dtype=torch.long)

n_masked_cols = MASK_COL_END - MASK_COL_START + 1
print(f'Context tokens : {len(context_indices)}  ({N_TEMPORAL} × {PATCH_GRID} × {PATCH_GRID - n_masked_cols})')
print(f'Target  tokens : {len(target_indices)}   ({N_TEMPORAL} × {PATCH_GRID} × {n_masked_cols})')
print(f'Mask pixel region: x=[{MASK_COL_START*PATCH_SIZE}, {(MASK_COL_END+1)*PATCH_SIZE})')

## 3. Load V-JEPA 2 Encoder + Predictor

Weights from `facebook/vjepa2-vitl-fpc64-256` — checkpoint key `target_encoder` (not `encoder`).
V-JEPA 2 predictor outputs **1024-dim** (same space as encoder) — compatible with the decoder.

In [ ]:
print('Loading V-JEPA 2 architecture ...')
result = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_1_vit_giant_384', pretrained=False, trust_repo=True)
encoder, predictor = result
print(f'encoder  : {type(encoder).__name__}')
print(f'predictor: {type(predictor).__name__}')

print(f'\nLoading weights from {VJEPA_WEIGHTS} ...')
ckpt = torch.load(VJEPA_WEIGHTS, map_location='cpu', weights_only=True)
print(f'Checkpoint keys: {list(ckpt.keys())}')

def load_sd(model, raw_sd):
    sd = {k.replace('module.', ''): v for k, v in raw_sd.items()}
    sd = {k.replace('backbone.', ''): v for k, v in sd.items()}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert len(missing) == 0 and len(unexpected) == 0, \
        f'missing={missing[:3]}  unexpected={unexpected[:3]}'

enc_key = 'target_encoder' if 'target_encoder' in ckpt else 'encoder'
load_sd(encoder,   ckpt[enc_key])
load_sd(predictor, ckpt['predictor'])

encoder.eval().to(DEVICE)
predictor.eval().to(DEVICE)
for p in list(encoder.parameters()) + list(predictor.parameters()):
    p.requires_grad_(False)

print(f'\nEncoder  : {sum(p.numel() for p in encoder.parameters())/1e6:.0f}M params')
print(f'Predictor: {sum(p.numel() for p in predictor.parameters())/1e6:.0f}M params')

## 4. Load Trained Decoder

In [ ]:
class UpsampleBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class LocalizationDecoder(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.upsample = nn.Sequential(
            UpsampleBlock(embed_dim, 512),
            UpsampleBlock(512, 256),
            UpsampleBlock(256, 128),
            UpsampleBlock(128, 64),
        )
        self.head = nn.Sequential(nn.Conv2d(64, 1, kernel_size=1), nn.Sigmoid())
    def forward(self, x): return self.head(self.upsample(x))

decoder = LocalizationDecoder().to(DEVICE)
ckpt_dec = torch.load(DECODER_CKPT, map_location='cpu', weights_only=True)
decoder.load_state_dict(ckpt_dec['decoder_state_dict'])
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)
print(f'Decoder loaded — val CE={ckpt_dec["val_ce"]:.1f}px  IoU={ckpt_dec["val_iou"]:.3f}  epoch={ckpt_dec["epoch"]}')

## 5. Inference Helpers

In [ ]:
SCALE_384_TO_256 = 1.0


def load_all_frames(clip_dir: Path) -> tuple[list, dict]:
    """Load every frame from a clip. Returns list of HxWx3 float32 arrays + metadata."""
    with open(clip_dir / 'metadata.json') as f:
        meta = json.load(f)
    frame_paths = sorted((clip_dir / 'frames').glob('*.png'))
    frames = [
        np.array(Image.open(p).resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)).astype(np.float32) / 255.0
        for p in frame_paths
    ]
    return frames, meta


def build_causal_window(all_frames: list, t_eval: int, window_size: int = NUM_FRAMES) -> torch.Tensor:
    """
    Return a [1, 3, window_size, H, W] video tensor using only frames ≤ t_eval.
    Pads the beginning by repeating the first frame when history is short.
    """
    start  = max(0, t_eval - window_size + 1)
    window = all_frames[start : t_eval + 1]
    if len(window) < window_size:
        window = [window[0]] * (window_size - len(window)) + window
    vid = np.stack(window)  # [T, H, W, 3]
    return torch.from_numpy(vid).permute(3, 0, 1, 2).unsqueeze(0)  # [1, 3, T, H, W]


def run_vjepa(video: torch.Tensor) -> torch.Tensor:
    """
    V-JEPA 2.1 ViT-G masked encoder + predictor pass.

    Encoder called twice:
    - training=True  -> 4-layer concat [1, N_ctx, 5632] for predictor input
    - training=False -> final layer    [1, N_ctx, 1408] for decoder context

    Predictor output is also 5632-dim (deep self-supervision multi-level);
    last 1408-dim chunk = final predictor block output, matching decoder's
    expected embedding space.
    """
    x = (video - IMAGENET_MEAN.unsqueeze(2)) / IMAGENET_STD.unsqueeze(2)
    x = x.to(DEVICE)

    ctx_mask = ctx_t.unsqueeze(0).to(DEVICE)   # [1, N_ctx]
    tgt_mask = tgt_t.unsqueeze(0).to(DEVICE)   # [1, N_tgt]

    with torch.no_grad():
        z_multi = encoder(x, masks=[ctx_mask], training=True)    # [1, N_ctx, 5632]
        z_final = encoder(x, masks=[ctx_mask], training=False)   # [1, N_ctx, 1408]
        z_pred, _ = predictor(z_multi, [ctx_mask], [tgt_mask], "video", 0)

    Zc     = z_final.squeeze(0).cpu()                 # [N_ctx, 1408]
    Zm_hat = z_pred.squeeze(0).cpu()[:, -EMBED_DIM:]  # [N_tgt, 1408] — last chunk of 5632-dim

    full = torch.zeros(N_TOTAL, EMBED_DIM)
    full[ctx_t] = Zc
    full[tgt_t] = Zm_hat

    per_temporal = full.view(N_TEMPORAL, N_SPATIAL, EMBED_DIM).permute(0, 2, 1)
    per_temporal = per_temporal.reshape(N_TEMPORAL, EMBED_DIM, PATCH_GRID, PATCH_GRID)
    return per_temporal


def decode_last(per_temporal: torch.Tensor) -> torch.Tensor:
    """Decode only the last temporal position (frames T-2 and T-1 of the window).
    Returns [1, H, W] heatmap."""
    with torch.no_grad():
        emb = per_temporal[-1].unsqueeze(0).to(DEVICE)
        return decoder(emb).squeeze(0).cpu()


print('Inference helpers defined.')


## 6. Run Inference on All Clips

In [ ]:
clip_dirs = sorted(BALL_CLIPS_DIR.iterdir())
print(f'Found {len(clip_dirs)} clips: {[d.name for d in clip_dirs]}')

results = {}

for clip_dir in clip_dirs:
    if not clip_dir.is_dir():
        continue
    name = clip_dir.name
    print(f'\n--- {name} ---')

    all_frames, meta = load_all_frames(clip_dir)
    total       = len(all_frames)
    eval_indices = np.linspace(0, total - 1, N_EVAL, dtype=int)
    print(f'  Total frames: {total}  eval at: {eval_indices}')

    heatmaps    = []
    eval_frames = []   # the actual frame at each evaluation point

    for t_eval in eval_indices:
        video_window = build_causal_window(all_frames, int(t_eval))
        per_temporal = run_vjepa(video_window)          # [N_TEMPORAL, D, 24, 24]
        hm           = decode_last(per_temporal)        # [1, H, W]
        heatmaps.append(hm)
        eval_frames.append(all_frames[int(t_eval)])

    heatmaps = torch.stack(heatmaps)   # [N_EVAL, 1, H, W]
    print(f'  Heatmaps: {heatmaps.shape}  max={heatmaps.max():.3f}')

    results[name] = {
        'heatmaps':    heatmaps,
        'meta':        meta,
        'eval_indices': eval_indices,
        'eval_frames': eval_frames,
    }


## 7. Visualise — Causal Trajectory

Each column = one evaluation point (causal window ending at that frame).  
Row 1: raw frame with mask box, GT ball centre (green +) and predicted peak (red +).  
Row 2: decoder heatmap from the last temporal position of the causal window.  
Row 3: overlay — heatmap blended into red channel.

In [ ]:
MASK_PX_X1 = MASK_COL_START * PATCH_SIZE
MASK_PX_X2 = (MASK_COL_END + 1) * PATCH_SIZE


def heatmap_peak(hm: torch.Tensor) -> tuple[int, int]:
    idx = hm.squeeze().argmax().item()
    return int(idx) % IMAGE_SIZE, int(idx) // IMAGE_SIZE


for name, res in results.items():
    heatmaps    = res['heatmaps']       # [N_EVAL, 1, H, W]
    meta        = res['meta']
    eval_idx    = res['eval_indices']
    eval_frames = res['eval_frames']    # list of HxWx3 arrays

    gt_lookup = {b['frame']: (b['cx_384'] * SCALE_384_TO_256, b['cy_384'] * SCALE_384_TO_256)
                 for b in meta['ball_centers'] if b.get('cx_384') is not None}

    fig, axes = plt.subplots(3, N_EVAL, figsize=(3 * N_EVAL, 10))
    fig.suptitle(
        f'{name}  — causal sliding window + predictor  (mask cols {MASK_COL_START}–{MASK_COL_END})',
        fontsize=12,
    )

    for col in range(N_EVAL):
        orig_idx = int(eval_idx[col])
        frame_np = eval_frames[col]
        hm_np    = heatmaps[col].squeeze().numpy()
        pred_cx, pred_cy = heatmap_peak(heatmaps[col])

        ax0 = axes[0, col]
        ax0.imshow(frame_np)
        rect = mpatches.Rectangle(
            (MASK_PX_X1, 0), MASK_PX_X2 - MASK_PX_X1, IMAGE_SIZE,
            linewidth=1.5, edgecolor='yellow', facecolor='yellow', alpha=0.15,
        )
        ax0.add_patch(rect)
        if orig_idx in gt_lookup:
            gx, gy = gt_lookup[orig_idx]
            ax0.plot(gx, gy, 'g+', ms=12, mew=2, label='GT')
        ax0.plot(pred_cx, pred_cy, 'r+', ms=12, mew=2, label='Pred')
        ax0.set_title(f'frame {orig_idx}', fontsize=7)
        ax0.axis('off')
        if col == 0:
            ax0.set_ylabel('Raw + mask', fontsize=8)

        ax1 = axes[1, col]
        ax1.imshow(hm_np, cmap='hot', vmin=0, vmax=1)
        ax1.axis('off')
        if col == 0:
            ax1.set_ylabel('Decoder\nheatmap', fontsize=8)

        ax2 = axes[2, col]
        overlay = frame_np.copy()
        overlay[:, :, 0] = np.clip(overlay[:, :, 0] + hm_np * 0.7, 0, 1)
        ax2.imshow(overlay)
        ax2.add_patch(mpatches.Rectangle(
            (MASK_PX_X1, 0), MASK_PX_X2 - MASK_PX_X1, IMAGE_SIZE,
            linewidth=1.5, edgecolor='yellow', facecolor='none',
        ))
        ax2.axis('off')
        if col == 0:
            ax2.set_ylabel('Overlay', fontsize=8)

    plt.tight_layout()
    fig.savefig(OUT_DIR / f'{name}_trajectory.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {OUT_DIR}/{name}_trajectory.png')


## 8. Quantitative Summary

Centre Error between predicted heatmap peak and GT ball position,
split by whether the ball is **inside** or **outside** the mask.
The inside-mask rows are the actual occlusion test.

In [ ]:
for name, res in results.items():
    heatmaps = res['heatmaps']
    meta     = res['meta']
    eval_idx = res['eval_indices']

    gt_lookup = {b['frame']: (b['cx_384'] * SCALE_384_TO_256, b['cy_384'] * SCALE_384_TO_256)
                 for b in meta['ball_centers'] if b.get('cx_384') is not None}

    ce_inside, ce_outside = [], []

    for col, t_eval in enumerate(eval_idx):
        orig_idx = int(t_eval)
        if orig_idx not in gt_lookup:
            continue
        gx, gy  = gt_lookup[orig_idx]
        px, py  = heatmap_peak(heatmaps[col])
        ce      = math.sqrt((px - gx)**2 + (py - gy)**2)
        in_mask = MASK_PX_X1 <= gx < MASK_PX_X2
        (ce_inside if in_mask else ce_outside).append(ce)

    def _fmt(lst):
        return f'{np.mean(lst):.1f}px  (n={len(lst)})' if lst else 'no GT'

    print(f'{name}:')
    print(f'  CE outside mask (visible)  : {_fmt(ce_outside)}')
    print(f'  CE inside  mask (occluded) : {_fmt(ce_inside)}')
